# Эксперименты с данными и парсингом репозиториев

Этот ноутбук предназначен для экспериментов с парсингом, анализом и обработкой репозиториев GitHub на языке Prolog. Здесь можно импортировать функции из data/github_prolog_repos.py и data/github_parser.py, а также сохранять результаты в папку data.

# Вес репозиториев на гите

In [1]:
import requests
from dotenv import load_dotenv
import os
import time
from typing import List, Dict

In [2]:
from github_prolog_repos import get_prolog_repos

In [3]:
import re
from typing import List

def is_complex_prolog(content: str, 
                      min_unique_predicates: int = 3,   # Минимум разных предикатов
                      min_rules_ratio: float = 0.05,     # Минимум 5% правил (не фактов)
                      min_complexity_markers: int = 2    # Минимум "сложных" конструкций
) -> bool:
    """
    Проверяет, является ли Prolog-код достаточно сложным 
    (не просто список фактов).
    """
    # Убираем комментарии и строки в кавычках для чистого анализа
    code = re.sub(r'%.*$', '', content, flags=re.MULTILINE)  # Комментарии
    code = re.sub(r"'[^']*'", '', code)  # Строковые литералы (упрощенно)
    
    lines = [l.strip() for l in code.split('\n') if l.strip() and not l.strip().startswith('%')]
    if not lines:
        return False
    
    # 1. Считаем уникальные предикаты (имя/арность, например: foo/3)
    predicates = set()
    for line in lines:
        # Ищем начало предиката: слово, за которым следует (
        match = re.match(r"^([a-z][a-z0-9_]*)\s*\(", line, re.IGNORECASE)
        if match:
            pred_name = match.group(1).lower()
            # Считаем арность по количеству запятых + 1 (упрощенно)
            arity = line.count(',') + 1 if '(' in line else 0
            predicates.add(f"{pred_name}/{arity}")
    
    # 2. Считаем правила (:-) и факты
    rules = sum(1 for l in lines if ':-' in l)
    facts = len(lines) - rules
    
    # 3. Ищем маркеры "настоящей" логики
    complexity_markers = 0
    markers = [
        r':-',           # Правила
        r'\bvar\(',      # Работа с переменными
        r'\bfindall\(',  # Сбор решений
        r'\bmaplist\(',  # Высшие порядки
        r'\b\+\+',       # Отрицание как неудача
        r'->',           # If-then
        r';',            # Дизъюнкция внутри правила
        r'\bis\s+\w',    # Арифметика (X is Y + 1)
        r'\brecursion',  # Рекурсия (в комментариях или именах)
        r'\b!\b',        # Cut
    ]
    for marker in markers:
        if re.search(marker, code):
            complexity_markers += 1
    
    # === Логика принятия решения ===
    
    # Если слишком мало разных предикатов — это скорее всего "дата", а не "код"
    if len(predicates) < min_unique_predicates:
        return False
    
    # Если это чистые факты (нет правил) — пропускаем
    if rules / max(len(lines), 1) < min_rules_ratio:
        return False
    
    # Если нет маркеров сложности — слишком просто
    if complexity_markers < min_complexity_markers:
        return False
    
    return True

In [7]:
import os
import requests
import base64
import time
from typing import List, Dict
import github_utils as gu
complex_repos = []
token = os.getenv("GITHUB_TOKEN")
headers_json = {  # Для JSON-ответов
    "Authorization": f"token {token}",
    "Accept": "application/vnd.github.v3+json"
}
d = get_prolog_repos(max_pages=10, per_page=30)  # ⚠️ per_page=100 может не сработать (макс 100, но лучше 30 для стабильности)


In [9]:
d[0]['full_name']

'nasa-jpl/open-source-rover'

In [11]:
for repo in d:
    cur = gu.download_github_repo(repo['full_name'],output_dir="repos")
    if cur is None:
        continue
    

AttributeError: module 'github_utils' has no attribute 'download_github_repo'